# Index Building with OGX & Milvus on RHOAI


#### Prerequesites

- run one foundational model on RHOAI
- run one embedding model on RHOAI
- run OGX Server in version >=0.7.1 on RHOAI (using remote::vllm providers) with enabled inline Milvus


Add environment variables
- **OGX_URL**
- **OGX_APIKEY** if auth/authz is enabled on OGX Server

NOTE: You can run also OGX locally, using either remote::vllm (for remote models) or ollama (for local models) providers.

### Install dependencies if needed

In [1]:
!pip install dotenv
!pip install pypdf
!pip install git+https://github.com/IBM/ai4rag.git@v0.5.4

Looking in indexes: https://pypi.org/simple/, https://pypi.org/simple/
Looking in indexes: https://pypi.org/simple/, https://pypi.org/simple/
Looking in indexes: https://pypi.org/simple/, https://pypi.org/simple/
  Cloning https://github.com/IBM/ai4rag.git (to revision v0.5.4) to /tmp/pip-req-build-j4hf1xij
  Running command git clone --filter=blob:none --quiet https://github.com/IBM/ai4rag.git /tmp/pip-req-build-j4hf1xij
  Running command git checkout -q 2d3f8ac49b505c994b5d8a12f5e8bfde53bbcc2b
  Resolved https://github.com/IBM/ai4rag.git to commit 2d3f8ac49b505c994b5d8a12f5e8bfde53bbcc2b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


#### Import dependencies

**NOTE:** Loads OGX_URL from `.env` file unless environment variables are pre-configured.

**NOTE:** Logging from httpx library set to WARNING level.

In [ ]:
import os

# Load from .env only if OGX_URL not already set
if not os.getenv('OGX_URL'):
    from dotenv import load_dotenv
    load_dotenv()

import logging
logging.getLogger("httpx").setLevel(logging.WARNING)

from llama_stack_client import LlamaStackClient

#### Create OGX Client object

**NOTE:** If you do not have authorization enabled on your OGX instance, just skip providing `OGX_APIKEY`.


In [ ]:
client = LlamaStackClient(
    base_url=os.getenv("OGX_URL", "http://localhost:5321"),
    api_key=os.getenv("OGX_APIKEY")
)

#### Models used in the example

In [4]:
# Read model names from environment variables with fallbacks
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "vllm-embedding/granite-278m-multilingual-1")
EMBEDDING_MODEL_DIMENSION = int(os.getenv("EMBEDDING_MODEL_DIMENSION", "768"))

LLM_MODEL = os.getenv("LLM_MODEL", "vllm-inference-llama-3-1/redhataillama-31-8b-instruct")

In [5]:
client.models.list()[0].model_dump()

{'id': 'vllm-embedding/nomic-embed-text-v1.5',
 'created': 1776949284,
 'owned_by': 'llama_stack',
 'custom_metadata': {'model_type': 'embedding',
  'provider_id': 'vllm-embedding',
  'provider_resource_id': 'nomic-ai/nomic-embed-text-v1.5',
  'embedding_dimension': 768},
 'object': 'model'}

#### Tests model availability via OGX

In [6]:
msgs = [{"role": "user", "content": "Generate a short poem"}]

llm_resp = client.chat.completions.create(
    messages=msgs,
    model=LLM_MODEL
)

llm_resp.model_dump()

{'id': 'chatcmpl-e5991b4f38a7454a9689bf7a844b49f5',
 'choices': [{'finish_reason': 'stop',
   'index': 0,
   'message': {'annotations': None,
    'audio': None,
    'content': "Softly falls the evening dew,\nA gentle hush, a peaceful hue,\nThe stars appear, one by one,\nA night of rest, for everyone.\n\nThe world is quiet, still and deep,\nThe moon's soft light, our souls do keep,\nIn this calm moment, we are free,\nA fleeting peace, for you and me.",
    'function_call': None,
    'refusal': None,
    'role': 'assistant',
    'tool_calls': None,
    'reasoning_content': None},
   'logprobs': None,
   'stop_reason': None}],
 'created': 1776949286,
 'model': 'vllm-inference/llama-3-2-3b',
 'object': 'chat.completion',
 'service_tier': None,
 'system_fingerprint': None,
 'usage': {'completion_tokens': 71,
  'completion_tokens_details': None,
  'prompt_tokens': 39,
  'prompt_tokens_details': None,
  'total_tokens': 110},
 'prompt_logprobs': None,
 'kv_transfer_params': None}

In [7]:
emb_response = client.embeddings.create(input="Hello", model=f"{EMBEDDING_MODEL}")
print(f"Embedding dimension: {len(emb_response.data[0].embedding)}")
print(f"Model: {emb_response.model}")

Embedding dimension: 768
Model: vllm-embedding/nomic-embed-text-v1.5


### Documents for knwoledge database building and benchmarking data

**NOTE:** The below example assumes that both documents for index building and benchmarking data resides in subfolder: `data/`. Moreover, this example itself does not deliver these artifacts - you need to replace:
- `documents_path` with list of paths to documents used for index building
- `benchmark_qa_path` with filename of benchmarking dataset, against which we'll test retrieval

In [8]:
from pathlib import Path
import json

In [9]:
docs_folder_path = Path("data/")
documents_path = [docs_folder_path.joinpath(doc) for doc in ("ibm-annual-report-2024-pt1_1-20.pdf", "ibm-annual-report-2024-pt2_20-40.pdf")]
benchmark_qa_path = docs_folder_path.joinpath("ibm_annual_report_pdf_benchmarking_data.json")

In [10]:
benchmark_qa = json.loads(benchmark_qa_path.read_bytes())
benchmark_qa[:3]

[{'correct_answer_document_ids': ['ibm-annual-report-2024-pt1_1-20.pdf'],
  'question': 'What was the operating other income and expense in 2024?',
  'correct_answer': '$1,656 million'},
 {'correct_answer_document_ids': ['ibm-annual-report-2024-pt1_1-20.pdf'],
  'question': 'What is the amount of gain on land/building dispositions included in "Other"',
  'correct_answer': '$126 million'},
 {'correct_answer_document_ids': ['ibm-annual-report-2024-pt1_1-20.pdf'],
  'question': 'How much was the non-operating retirement-related costs in the current-year period?',
  'correct_answer': '$3,457 million'}]

## Index building

### 1.0 Define helper objects

In [11]:
from pathlib import Path
from typing import Sequence
from functools import lru_cache
import io

from pypdf import PdfReader
from langchain_core.documents import Document


def _txt_to_string(binary_data: bytes) -> str:
    return binary_data.decode("utf-8", errors="ignore")


def _pdf_to_string(binary_data: bytes) -> str:
    with io.BytesIO(binary_data) as open_pdf_file:
        reader = PdfReader(open_pdf_file)
        full_text = [page.extract_text() for page in reader.pages]
        return "\n".join(full_text)


class FileStoreException(Exception):
    pass


class FileStore:
    """
    Class used to load locally saved input files.
    """

    suffix_to_func = {
        ".txt": _txt_to_string,
        ".pdf": _pdf_to_string,
    }

    def __init__(self, path: str | Path | Sequence[str] | Sequence[Path]):
        self.path = Path(path)
        self.is_dir = path.is_dir()
        self.files = {}

    def __repr__(self) -> str:
        ret = f"{self.__class__.__name__}(path={self.path})"
        return ret

    def load_as_documents(self) -> list[Document]:
        """Read files as langchain documents"""
        contents = self._load_content()
        documents = [Document(page_content=content[0], metadata={"document_id": content[1]}) for content in contents]

        return documents

    @lru_cache(maxsize=2)
    def _load_content(self) -> list[tuple[str, str]]:
        """Load file(s) from given path"""
        if self.is_dir:
            contents = [(self._read_single_content(file), file.name) for file in self.path.iterdir()]
            return contents

        return [(self._read_single_content(self.path), self.path.name)]

    def _process_file(self, filepath: Path, file_content: bytes | None) -> str:
        """
        Extracts text from bytes for various file types.

        Parameters
        ----------
        filepath : Path
            Path to file

        file_content : bytes | None
            File content as bytes

        Returns
        -------
        str
            Extracted file text
        """

        handler = self.suffix_to_func.get(filepath.suffix, None)

        try:
            text = handler(file_content)
        except Exception as exc:
            raise FileStoreException(f"Failed to load file.") from exc

        return text

    def _read_single_content(self, filepath: Path) -> str:
        """Read single file"""
        with open(filepath, "rb") as file:
            content = file.read()
        text = self._process_file(filepath=filepath, file_content=content)
        self.files[str(filepath)] = text
        return text

### 1.1  Load documents in LangChain represenation

In [12]:
file_store = FileStore(Path("data/docs"))
documents = file_store.load_as_documents()

In [13]:
print(f"Loaded {len(documents)} documents")
print(f"First document ID: {documents[0].metadata['document_id']}")

Loaded 2 documents
First document ID: ibm-annual-report-2024-pt1_1-20.pdf


### 1.2 Define OGX embedding model object

In [14]:
from ai4rag.rag.embedding.llama_stack import LSEmbeddingModel

embedding_model = LSEmbeddingModel(
    model_id=EMBEDDING_MODEL,
    client=client,
    params=dict(
        provider_id="milvus-remote",
        embedding_dimension=EMBEDDING_MODEL_DIMENSION,
        context_length=512,
    )
    
)

### 1.3 Define OGX Vector Store object

In [15]:
from ai4rag.rag.vector_store.llama_stack import LSVectorStore

vector_store = LSVectorStore(
    embedding_model=embedding_model,
    client=client,
    provider_id="milvus-remote",

)

In [16]:
vector_store._ls_vs.id

'vs_cb0bb787-ddec-40bb-8bab-b51ccf08fec5'

### 1.4 Chunk and embed documents into vector store

In [17]:
from ai4rag.rag.chunking import LangChainChunker

chunking_method = "recursive"
chunk_size = 256
chunk_overlap = 64

chunker = LangChainChunker(method=chunking_method, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
chunked_documents = chunker.split_documents(documents)
vector_store.add_documents(chunked_documents)

### 1.5 Test retrieval

In [18]:
question = benchmark_qa[-1].get("question")
question

'What was the total revenue for year 2024?'

In [19]:
search_response = client.vector_stores.search(
    vector_store_id=vector_store._ls_vs.id,
    query=question,
    max_num_results=5,
)
search_response

VectorStoreSearchResponse(data=[Data(content=[DataContent(text='Financial Performance Summary                        \nIn 2024, we reported $62.8 billion in revenue, income from continuing operations of $6.0 billion, which includes the impact of the', type='text', chunk_metadata=None, embedding=None, metadata=None)], file_id='ibm-annual-report-2024-pt1_1-20.pdf', filename='', score=0.8112797141075134, attributes={'document_id': 'ibm-annual-report-2024-pt1_1-20.pdf', 'start_index': 29913.0, 'sequence_number': 166.0}), Data(content=[DataContent(text='Transaction Processing revenue of $8,277 million increased 8.7 percent as reported (9.6 percent adjusted for currency) in 2024', type='text', chunk_metadata=None, embedding=None, metadata=None)], file_id='ibm-annual-report-2024-pt1_1-20.pdf', filename='', score=0.7954844236373901, attributes={'document_id': 'ibm-annual-report-2024-pt1_1-20.pdf', 'start_index': 68073.0, 'sequence_number': 388.0}), Data(content=[DataContent(text='billion in 20